# 04 — Isolation Forest

**Purpose:** Train the unsupervised anomaly detector on benign-only traffic.  
**Acceptance criterion:** recall ≥ 50% AND FP rate ≤ 10% **simultaneously** on val set.  
Meeting only one is a failure.

**Rule R4:** IF trained exclusively on benign samples from the training set.  
Never trained on attack samples.

**Rule R5:** recall and FP rate are always reported together. Never one without the other.

**Input:** `training/splits/train.parquet` (benign rows only), `training/splits/val.parquet`  
**Output:** `training/models/if_vN.pkl`, `training/results/if_val_report.txt`

## 1. Load Benign-Only Training Set

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

SPLITS  = Path("../../training/splits")
RESULTS = Path("../../training/results")
RESULTS.mkdir(parents=True, exist_ok=True)

META_COLS = [
    "sample_id", "timestamp", "_source", "_row_hash",
    "status_code", "req_count_1s", "req_count_5s", "req_count_60s",
    "error_rate_4xx_60s", "endpoint_diversity_60s",
]

train = pd.read_parquet(SPLITS / "train.parquet")
train.drop(columns=[c for c in META_COLS if c in train.columns], inplace=True)

y_train = train.pop("label")
X_train = train

X_benign = X_train[y_train == "benign"].copy()
print(f"Benign-only training samples: {X_benign.shape[0]:,}")
print(f"Feature columns: {X_benign.shape[1]}")
print("CRITICAL: IF trains on benign only — never on attack samples.")

Benign-only training samples: 69,379
Feature columns: 66
CRITICAL: IF trains on benign only — never on attack samples.


## 2. Train Isolation Forest

In [2]:
from sklearn.ensemble import IsolationForest
import joblib

# contamination = estimated fraction of anomalies in production traffic
# Conservative estimate: 5% (1 in 20 requests is malicious)
# Adjust based on real traffic data if available
CONTAMINATION = 0.05

iso = IsolationForest(
    n_estimators=200,
    contamination=CONTAMINATION,
    max_samples="auto",
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_benign)
print(f"IF trained on {X_benign.shape[0]:,} benign samples.")
print(f"contamination={CONTAMINATION} — adjust in Section 3 if needed.")

IF trained on 69,379 benign samples.
contamination=0.05 — adjust in Section 3 if needed.


## 3. Calibrate contamination/threshold on Val

In [3]:
val = pd.read_parquet(SPLITS / "val.parquet")
val.drop(columns=[c for c in META_COLS if c in val.columns], inplace=True)
y_val = val.pop("label")
X_val = val

scores   = iso.decision_function(X_val)
y_binary = (y_val != "benign").astype(int)  # 1=attack, 0=benign

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

thresholds = np.linspace(scores.min(), scores.max(), 300)
recalls, fp_rates = [], []

for t in thresholds:
    predicted_attack = (scores < t).astype(int)
    tp = ((predicted_attack == 1) & (y_binary == 1)).sum()
    fp = ((predicted_attack == 1) & (y_binary == 0)).sum()
    fn = ((predicted_attack == 0) & (y_binary == 1)).sum()
    tn = ((predicted_attack == 0) & (y_binary == 0)).sum()
    recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    fp_rates.append(fp / (fp + tn) if (fp + tn) > 0 else 0)

valid = [
    (t, r, f)
    for t, r, f in zip(thresholds, recalls, fp_rates)
    if r >= 0.50 and f <= 0.08
]

if not valid:
    print("WARNING: No threshold satisfies recall>=0.50 AND FP<=0.08")
    print("Review contamination parameter or feature quality.")
    BEST_THRESHOLD = None
else:
    best = max(valid, key=lambda x: x[1])
    BEST_THRESHOLD, BEST_RECALL, BEST_FP = best
    print(f"Selected threshold : {BEST_THRESHOLD:.4f}")
    print(f"Recall             : {BEST_RECALL:.4f}")
    print(f"FP rate            : {BEST_FP:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(fp_rates, recalls, lw=2)
plt.axvline(x=0.08, color="red",   linestyle="--", label="FP limit = 0.08")
plt.axhline(y=0.50, color="green", linestyle="--", label="Recall floor = 0.50")
if BEST_THRESHOLD is not None:
    plt.scatter(
        [BEST_FP], [BEST_RECALL], color="black", zorder=5,
        label=f"Selected (recall={BEST_RECALL:.2f}, FP={BEST_FP:.2f})",
    )
plt.xlabel("False Positive Rate")
plt.ylabel("Recall (Attack Detection Rate)")
plt.title("Isolation Forest — Recall vs FP Rate")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS / "if_recall_fp_curve.png", dpi=150)
plt.close()

Selected threshold : 0.0287
Recall             : 0.7605
FP rate            : 0.0784


## 4. Recall vs FP Rate Curve

In [4]:
assert BEST_THRESHOLD is not None, (
    "GATE FAILED: No valid threshold found. "
    "IF does not meet recall>=0.50 AND FP<=0.08 simultaneously. "
    "Do not proceed to ONNX export."
)

predicted_attack_final = (scores < BEST_THRESHOLD).astype(int)
tp = ((predicted_attack_final == 1) & (y_binary == 1)).sum()
fp = ((predicted_attack_final == 1) & (y_binary == 0)).sum()
fn = ((predicted_attack_final == 0) & (y_binary == 1)).sum()
tn = ((predicted_attack_final == 0) & (y_binary == 0)).sum()

final_recall  = tp / (tp + fn)
final_fp_rate = fp / (fp + tn)

print("=== Isolation Forest — Final Metrics (Validation Set) ===")
print(f"Recall (attack detection rate) : {final_recall:.4f}")
print(f"False Positive rate            : {final_fp_rate:.4f}")
print(f"Threshold                      : {BEST_THRESHOLD:.4f}")
print("")
print("Both metrics always reported together — R5.")

=== Isolation Forest — Final Metrics (Validation Set) ===
Recall (attack detection rate) : 0.7605
False Positive rate            : 0.0784
Threshold                      : 0.0287

Both metrics always reported together — R5.


## 5. Threshold Selection

In [5]:
import json

MODELS = Path("../../training/models")
MODELS.mkdir(parents=True, exist_ok=True)

joblib.dump(iso, MODELS / "if_v5.pkl")

metadata = {
    "threshold":     float(BEST_THRESHOLD),
    "contamination": CONTAMINATION,
    "trained_on":    "benign_only",
    "val_recall":    float(final_recall),
    "val_fp_rate":   float(final_fp_rate),
}
with open(MODELS / "if_v5_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved: {MODELS / 'if_v5.pkl'}")
print(f"Saved: {MODELS / 'if_v5_metadata.json'}")

Saved: ../../training/models/if_v5.pkl
Saved: ../../training/models/if_v5_metadata.json


## 6. Final Metrics (Recall AND FP Rate Together — Never One Without the Other)

## 6. Metric Reporting Rule (R5)

The Isolation Forest is evaluated on TWO metrics simultaneously.
Meeting only one is a failure:

| Metric | Requirement | Measured |
|--------|-------------|----------|
| Recall (attack detection) | ≥ 0.50 | see Section 4 |
| False Positive Rate | ≤ 0.10 | see Section 4 |

These numbers are always reported together in the thesis.
Never cite recall without FP rate. Never cite FP rate without recall.